# Fase 2 — Audit degli schemi `query.csv` / `record.csv`

Legge **esclusivamente le intestazioni CSV** delle sorgenti contenute in `Market.zip` e degli eventuali file alla root di Drive. NON legge righe, non stampa classi attribuite ai casi, non interroga LLM e non esegue scoring. Il risultato serve a progettare l'audit separato per la distribuzione delle quattro classi riservate.

Eseguire solo dopo i test PyCharm del modulo `test_ood_source_schema_audit.py`.


In [ ]:
# FASE 2 — schema-only, legge soltanto le intestazioni CSV. ZERO API/GT per-caso.
from google.colab import drive
from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
DATA = Path("/content/drive/MyDrive/Tesi_OpenRCA")
BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"

drive.mount("/content/drive")
if not (DATA / "Market.zip").is_file():
    raise FileNotFoundError("Market.zip assente nella cartella Tesi_OpenRCA")

if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Clone OOD non valido: non sovrascrivo il contenuto")
    current = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    if current != BRANCH:
        raise RuntimeError(f"Branch inatteso {current}: nessuno switch automatico")
    status = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip()
    if status:
        raise RuntimeError("Clone con modifiche: non applico pull")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True, capture_output=True, text=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch",
         "--branch", BRANCH, URL, str(REPO)],
        check=True, capture_output=True, text=True
    )

head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("=== FASE 2 — ISPEZIONE SCHEMA, NON ETICHETTE ===")
print("Branch:", BRANCH)
print("Commit:", head)

env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
result = subprocess.run(
    [sys.executable, "-m", "evaluation.ood_source_schema_audit",
     "--data-dir", str(DATA)],
    cwd=str(REPO), env=env, text=True, capture_output=True
)
if result.returncode:
    raise RuntimeError("Schema audit fallito:\n" + result.stderr)
try:
    data = json.loads(result.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("Non è stato restituito un JSON verificabile") from exc
if (
    data.get("audit") != "ood_source_schema_only_v1"
    or data.get("read_records") is not False
    or data.get("read_ground_truth_values") is not False
    or data.get("read_telemetry") is not False
    or data.get("model_api_calls") != 0
    or data.get("files_written") is not False
):
    raise RuntimeError("Schema audit restituisce indicatori non conformi")

print("\n=== SCHEMI FILE E RELATIVE POSIZIONI ===")
print(json.dumps(data, indent=2, ensure_ascii=False))
print("\nNESSUNA etichetta per incidente consultata o mostrata; ZERO API LLM.")


## Dopo l'esecuzione

Condividere **solo l'output**. Lo schema ci consentirà di verificare quali fonti rappresentano cloudbed-1 e cloudbed-2 prima di scrivere uno script che conta le classi senza diffondere ground truth nelle diagnosi.
